# the plan is to use this embedding model
Embedding → Bi-LSTM(64) → Bi-LSTM(64) → Dense(1)


the model and vectorizer configs are available in /model folder (D:\AMAZONML\artifacts\approach8\model)

i need you to fine tune this model on our dataset (using only the product_catalog and price column)


the train dataset is in :

D:\AMAZONML\dataset\train.csv


perform an 80-20 train-val split on the train dataset
and then evaluate MSE,MAE, SMAPE (main metric), R2 on the validation dataset


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv(r'/Users/gabi/Desktop/DL stuff/AMAZONML/student_resource/dataset/train.csv')

# Select relevant columns
df = df[['catalog_content', 'price']]
df = df.dropna()

# Split the data
X = df['catalog_content']
y = df['price']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
from keras.saving import load_model
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
import json

# Load the model
model_path = r'/Users/gabi/Desktop/DL stuff/AMAZONML/approach8/model/saved_model.keras'
model = load_model(model_path)

# Load the vectorizer config
vectorizer_config_path = r'/Users/gabi/Desktop/DL stuff/AMAZONML/approach8/model/vectorizer_config.json'
with open(vectorizer_config_path, 'r') as f:
    vectorizer_config = json.load(f)

# Fetch the vectorizer layer embedded in the model and restore its vocabulary
vectorizer = model.get_layer('text_vectorization')
if 'vocab' in vectorizer_config:
    vectorizer.set_vocabulary(vectorizer_config['vocab'])

model.summary()


In [ ]:
# Prepare tf.data datasets
BATCH_SIZE = 32
EPOCHS = 10
EARLY_STOP_PATIENCE = 2

train_ds = tf.data.Dataset.from_tensor_slices((X_train.values, y_train.values))
train_ds = train_ds.shuffle(buffer_size=len(X_train)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val.values, y_val.values))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Compile the model
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=EARLY_STOP_PATIENCE,
        restore_best_weights=True
    )
]

# Train the model
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks
)


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# 1. Make predictions on the validation set
y_val_true = y_val.values
y_pred = model.predict(val_ds).flatten()

# 2. Calculate evaluation metrics
mse = mean_squared_error(y_val_true, y_pred)
mae = mean_absolute_error(y_val_true, y_pred)
r2 = r2_score(y_val_true, y_pred)

def smape(y_true, y_pred):
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    # avoid division by zero
    mask = denominator != 0
    smape_values = np.zeros_like(denominator)
    smape_values[mask] = numerator[mask] / denominator[mask]
    return np.mean(smape_values) * 100

smape_score = smape(y_val_true, y_pred)


# 3. Print the evaluation metrics
print("Evaluation metrics on the validation set:")
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"SMAPE: {smape_score:.4f}%")
print(f"R2 Score: {r2:.4f}")
